### Giải thích Bước 1: Tích chập 1 chiều (1D Convolution)

**Ý nghĩa vật lý & ứng dụng:**
- Tích chập 1D thường được áp dụng cho các dữ liệu dạng chuỗi thời gian (time-series) như tín hiệu âm thanh, dữ liệu cảm biến, hoặc văn bản (NLP).
- Trong ví dụ trên, chúng ta sử dụng một bộ lọc trung bình trượt (Moving Average Filter) có kích thước bằng 3: `w_1d = [1/3, 1/3, 1/3]`. Bộ lọc này có tác dụng làm mịn (smooth) tín hiệu đầu vào, giảm bớt nhiễu tần số cao.

**Cơ chế hoạt động của hàm `conv1d`:**
1. **Padding (Đệm):** Thêm các giá trị `0` vào hai đầu của mảng đầu vào để bảo toàn kích thước hoặc kiểm soát kích thước đầu ra.
2. **Trượt bộ lọc (Sliding window):** Bộ lọc `w` dịch chuyển dọc theo mảng đã đệm với bước nhảy `stride`.
3. **Nhân chập & Cộng tích lũy:** Tại mỗi vị trí dừng, ta nhân từng phần tử tương ứng giữa vùng cửa sổ tín hiệu và bộ lọc, cộng tổng lại rồi cộng thêm hệ số chệch `bias` (nếu có).

### Giải thích Bước 2: Tích chập 2 chiều đơn kênh (2D Convolution - Single Channel)

**Ý nghĩa vật lý & ứng dụng:**
- Tích chập 2D là nền tảng cốt lõi của mạng nơ-ron tích chập (CNN) xử lý ảnh xám (grayscale) hoặc các bản đồ đặc trưng đơn kênh.
- Trong ví dụ này, ảnh đầu vào là một ma trận $5 \times 5$ giả lập một vệt sáng thẳng đứng ở nửa bên trái (các giá trị `10`) và vùng tối ở nửa bên phải (các giá trị `0`).
- Bộ lọc dò cạnh dọc (Vertical Edge Filter) kích thước $3 \times 3$:
  $$\begin{bmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{bmatrix}$$
  khi quét qua ảnh sẽ cho giá trị âm hoặc dương cực đại tại những nơi có sự thay đổi đột ngột về cường độ sáng theo chiều ngang (phát hiện biên cạnh dọc).

**Cơ chế hoạt động của hàm `conv2d_11`:**
- Hàm duyệt qua hai chiều không gian (chiều cao $H$ và chiều rộng $W$).
- Tại mỗi vùng cửa sổ kích thước $3 \times 3$ trên ảnh, thực hiện phép nhân Hadamard (nhân từng phần tử tương ứng) với bộ lọc, sau đó tính tổng tất cả để cho ra một pixel duy nhất trên ảnh kết quả `img_edge`.

### Giải thích Bước 3: Tích chập 2 chiều tổng quát (Multi-channel & Batch)

**Ý nghĩa vật lý & ứng dụng:**
- Trong thực tế sâu (Deep Learning), dữ liệu ảnh thường có nhiều kênh màu (ví dụ RGB có 3 kênh) và được truyền vào theo từng lô (Batch) để tối ưu hóa tính toán trên GPU/CPU.
- Đầu ra của một lớp tích chập cũng thường bao gồm nhiều kênh (tương ứng với số lượng bộ lọc khác nhau được áp dụng).

**Cấu trúc các ma trận đầu vào và đầu ra:**
- **Đầu vào $A$:** Kích thước `(m, in_height, in_width, in_channel)` = `(1, 4, 4, 3)` (1 ảnh RGB kích thước $4 \times 4$).
- **Bộ lọc $W$:** Kích thước `(f, f, in_channel, out_channel)` = `(3, 3, 3, 2)` (2 bộ lọc kích thước $3 \times 3$, mỗi bộ lọc phải có số kênh bằng số kênh đầu vào là 3).
- **Biases $b$:** Gồm 2 giá trị bias tương ứng với 2 bộ lọc đầu ra.
- **Đầu ra:** Kích thước thu được là `(1, 2, 2, 2)` tương ứng với `(batch_size, out_height, out_width, out_channels)`.

**Cơ chế hoạt động của hàm `conv2d`:**
- Hàm sử dụng 4 vòng lặp lồng nhau để duyệt qua: từng mẫu trong batch (`m`), từng tọa độ pixel đầu ra (`h`, `w`), và từng kênh đầu ra (`c`).
- Tại mỗi bước, một khối 3D trên ảnh đầu vào (kích thước $f \times f \times in\_channel$) sẽ được nhân chập với bộ lọc tương ứng của kênh đầu ra đó và cộng thêm bias.

In [ ]:
# 1
import numpy as np

# Cho phép in mảng numpy đẹp và dễ nhìn hơn
np.set_printoptions(precision=3, suppress=True)

# =====================================================================
# 1. TÍCH CHẬP 1 CHIỀU (1D CONVOLUTION)
# =====================================================================

def conv1d(a, w, b=0, stride=1, pad=0):
    """
    Tính tích chập 1 chiều (có bias)
    """
    w_old = a.shape[0]
    f = w.shape[0]
    a_pad = np.pad(a, pad_width=pad, mode='constant', constant_values=0)
    w_new = int((w_old - f + 2 * pad) / stride) + 1
    a_res = np.zeros((w_new))
    for i in range(w_new):
        start = i * stride
        end = start + f
        a_res[i] = np.sum(a_pad[start:end] * w) + b
    return a_res

print("=== 1. KẾT QUẢ TÍCH CHẬP 1D ===")
# Tạo tín hiệu sin 1D gồm 10 phần tử
x_clean = np.sin(np.arange(10) / 2.0)
w_1d = (1.0 / 3.0) * np.ones(3) # Bộ lọc trung bình kích thước 3
x_filtered = conv1d(x_clean, w_1d, pad=1)

print("Tín hiệu đầu vào x_clean (10 phần tử):")
print(x_clean)
print("\nKết quả sau khi qua bộ lọc 1D (conv1d):")
print(x_filtered)
print("\n" + "="*50 + "\n")


# =====================================================================
# 2. TÍCH CHẬP 2 CHIỀU ĐƠN KÊNH (2D CONVOLUTION - SINGLE CHANNEL)
# =====================================================================

def conv2d_11(A, W, b=0, stride=1, pad=0):
    """
    A: input, A.shape = (n_H_old, n_W_old)
    W: filter, W.shape = (f, f)
    """
    n_H_old, n_W_old = A.shape
    f, f = W.shape
    A_pad = np.pad(A, pad_width=pad, mode='constant', constant_values=0)

    n_H_new = int((n_H_old - f + 2 * pad) / stride) + 1
    n_W_new = int((n_W_old - f + 2 * pad) / stride) + 1

    A_res = np.zeros((n_H_new, n_W_new))

    for h in range(n_H_new):
        for v in range(n_W_new):
            h_start = h * stride
            h_end = h_start + f
            v_start = v * stride
            v_end = v_start + f
            A_res[h, v] = np.sum(A_pad[h_start:h_end, v_start:v_end] * W) + b

    return A_res

print("=== 2. KẾT QUẢ TÍCH CHẬP 2D ĐƠN KÊNH ===")
# Giả lập 1 bức ảnh xám kích thước 5x5
img_gray = np.array([
    [10, 10, 10, 0, 0],
    [10, 10, 10, 0, 0],
    [10, 10, 10, 0, 0],
    [10, 10, 10, 0, 0],
    [10, 10, 10, 0, 0]
], dtype=float)

# Bộ lọc dò cạnh thẳng đứng (Vertical edge detection)
w_vert = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
], dtype=float)

img_edge = conv2d_11(img_gray, w_vert, stride=1, pad=0)

print("Ảnh đầu vào 5x5 (có sọc dọc ở giữa):")
print(img_gray)
print("\nBộ lọc dọc 3x3:")
print(w_vert)
print("\nKết quả phát hiện cạnh dọc (conv2d_11):")
print(img_edge)
print("\n" + "="*50 + "\n")


# =====================================================================
# 3. TÍCH CHẬP 2 CHIỀU TỔNG QUÁT (2D CONVOLUTION - MULTI-CHANNEL / BATCH)
# =====================================================================

def conv2d(A, W, b, stride=1, pad=0):
    """
    A: input, A.shape = (m, in_height, in_width, in_channel)
    W: filters, W.shape = (f, f, in_channel, out_channel)
    b: biases, b.shape = (out_channel)
    """
    assert A.shape[3] == W.shape[2], \
        'number of input channels ({}) in A != number of input channels ({}) in W'.format(A.shape[3], W.shape[2])

    m, in_height, in_width, _ = A.shape
    f, _, in_channel, out_channel = W.shape

    A_pad = np.pad(A, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode='constant', constant_values=0)

    out_height = int((in_height - f + 2 * pad) / stride) + 1
    out_width = int((in_width - f + 2 * pad) / stride) + 1

    A_res = np.zeros((m, out_height, out_width, out_channel))

    for i in range(m):
        for h in range(out_height):
            for w in range(out_width):
                for c in range(out_channel):
                    h_start = h * stride
                    h_end = h_start + f
                    w_start = w * stride
                    w_end = w_start + f

                    a_slide = A_pad[i, h_start:h_end, w_start:w_end, :]
                    A_res[i, h, w, c] = np.sum(a_slide * W[:, :, :, c]) + b[c]

    return A_res

print("=== 3. KẾT QUẢ TÍCH CHẬP 2D TỔNG QUÁT (ĐA KÊNH & BATCH) ===")
# Giả lập batch m=1, ảnh RGB 4x4x3
A_batch = np.ones((1, 4, 4, 3))
# Tạo 2 bộ lọc kích thước 3x3x3
W_filters = np.ones((3, 3, 3, 2))
b_bias = np.array([1.0, 2.0])

out_batch = conv2d(A_batch, W_filters, b_bias, stride=1, pad=0)

print(f"Kích thước đầu vào A : {A_batch.shape} (batch_size=1, height=4, width=4, channels=3)")
print(f"Kích thước bộ lọc W  : {W_filters.shape} (f_h=3, f_w=3, in_chan=3, out_chan=2)")
print(f"Kích thước đầu ra    : {out_batch.shape}")
print("\nGiá trị ma trận đầu ra tại kênh 0 (sau khi cộng bias 1.0):")
print(out_batch[0, :, :, 0])
print("\nGiá trị ma trận đầu ra tại kênh 1 (sau khi cộng bias 2.0):")
print(out_batch[0, :, :, 1])

=== 1. KẾT QUẢ TÍCH CHẬP 1D ===
Tín hiệu đầu vào x_clean (10 phần tử):
[ 0.     0.479  0.841  0.997  0.909  0.598  0.141 -0.351 -0.757 -0.978]

Kết quả sau khi qua bộ lọc 1D (conv1d):
[ 0.16   0.44   0.773  0.916  0.835  0.55   0.13  -0.322 -0.695 -0.578]


=== 2. KẾT QUẢ TÍCH CHẬP 2D ĐƠN KÊNH ===
Ảnh đầu vào 5x5 (có sọc dọc ở giữa):
[[10. 10. 10.  0.  0.]
 [10. 10. 10.  0.  0.]
 [10. 10. 10.  0.  0.]
 [10. 10. 10.  0.  0.]
 [10. 10. 10.  0.  0.]]

Bộ lọc dọc 3x3:
[[-1.  0.  1.]
 [-1.  0.  1.]
 [-1.  0.  1.]]

Kết quả phát hiện cạnh dọc (conv2d_11):
[[  0. -30. -30.]
 [  0. -30. -30.]
 [  0. -30. -30.]]


=== 3. KẾT QUẢ TÍCH CHẬP 2D TỔNG QUÁT (ĐA KÊNH & BATCH) ===
Kích thước đầu vào A : (1, 4, 4, 3) (batch_size=1, height=4, width=4, channels=3)
Kích thước bộ lọc W  : (3, 3, 3, 2) (f_h=3, f_w=3, in_chan=3, out_chan=2)
Kích thước đầu ra    : (1, 2, 2, 2)

Giá trị ma trận đầu ra tại kênh 0 (sau khi cộng bias 1.0):
[[28. 28.]
 [28. 28.]]

Giá trị ma trận đầu ra tại kênh 1 (sau khi cộng bia